# CT Scans Report Generation (Inference) — Cleaned Version


This notebook provides a compact, reproducible inference pipeline that converts thoracic CT volumes into radiology-style Findings & Impressions text. It loads the pretrained CT visual encoder (CT-ViT), a learned projection head, and a LoRA-adapted LLM to produce report text from extracted CT features — all optimized to run on limited GPU resources (Colab T4, 16GB logical). The notebook contains only the core steps required for reliable inference: environment setup, model loading, feature extraction, report generation, and minimal postprocessing.

**NOTE**: CT scans can be quite large (around 400 MB each), which makes processing them for predictions a bit challenging, especially with limited computational resources. I did my best to make the model work for generating predictions on these large files, but there are still some limitations due to the size and complexity of the data.

Mount Google Drive, Hugging Face login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import login
login(token="#")


Install dependencies

In [ ]:
!pip install --quiet \
  transformers bitsandbytes accelerate peft datasets \
  nibabel torch torchvision tqdm huggingface_hub


Imports

In [ ]:
import os, torch, nibabel as nib
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model

Vision model and LoRA base

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

repo_id = "ibrahimhamamci/CT-RATE"
repo_type = "dataset"
base_local_dir = "ct_weights"

files_to_download = [
    "models/CT-CLIP-Related/CT-CLIP_v2.pt",
    "models/CT-CHAT/llama_3.1_8b/adapter_config.json",
    "models/CT-CHAT/llama_3.1_8b/adapter_model.safetensors",
]
for file_path in files_to_download:
    # Download into a temporary directory
    temp_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type=repo_type,
        local_dir="temp_hf_download",
        local_dir_use_symlinks=False
    )

    # Reconstruct original folder structure in base_local_dir
    full_dest_path = os.path.join(base_local_dir, file_path)
    os.makedirs(os.path.dirname(full_dest_path), exist_ok=True)
    shutil.move(temp_path, full_dest_path)

    print(f"Downloaded and moved: {full_dest_path}")

In [ ]:
!git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git /content/CT-CLIP
import sys, os
paths = [
    "/content/CT-CLIP",                    # for CT_CLIP.ctvit
    "/content/CT-CLIP/transformer_maskgit",# transformer lib
]
for p in paths:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing path: {p}")
sys.path.extend(paths)
print("Paths set:", paths)


In [ ]:
%cd /content/CT-CLIP/transformer_maskgit
!pip install -e .
%cd /content

Initialize and Load CTViT Model

In [ ]:
from transformer_maskgit.ctvit import CTViT
import torch

# Instantiate on CPU
ct_vit = CTViT(
    dim=512, codebook_size=8192, image_size=480,
    patch_size=24, temporal_patch_size=12,
    spatial_depth=4, temporal_depth=4,
    dim_head=32, heads=8, channels=1,
    use_vgg_and_gan=False
).eval().cpu()

# Load checkpoint (map to CPU first)
sd = torch.load("/content/ct_weights/models/CT-CLIP-Related/CT-CLIP_v2.pt", map_location="cpu")
sd = sd.get("state_dict", sd)
sd = {k.replace("module.",""):v for k,v in sd.items()}
ct_vit.load_state_dict(sd, strict=False)

# Move to GPU
ct_vit = ct_vit.cuda()
print(" CTViT ready on", next(ct_vit.parameters()).device)


**Function to Extract Features from CT Volumes Using CTViT**

-Loads a 3D CT scan volume using nibabel.

-Normalizes intensity values to a 0-1 range (clamping and scaling).

-Adds batch and channel dimensions, moves data to GPU.

-Resizes volume to fixed spatial and slice dimensions with trilinear interpolation.

-Passes the volume through the CTViT model to get encoded tokens (features).

-Reshapes and flattens tokens, then moves them back to CPU.

-Returns the extracted features as a tensor.

In [ ]:
import nibabel as nib
import torch
import torch.nn.functional as F

def extract_ctvit_features(ct_vit_model, volume_path,
                           max_slices=12, size=480):
    # Load & normalize on CPU
    vol = nib.load(volume_path).get_fdata().astype("float32")
    vol = torch.clamp(torch.from_numpy(vol), -1000.0, 400.0)
    vol = (vol + 1000.0) / 1400.0

    # To [1,1,D,H,W] → GPU
    x = vol.unsqueeze(0).unsqueeze(0).cuda()
    x = F.interpolate(x, size=(max_slices, size, size),
                      mode="trilinear", align_corners=False)

    # Forward through CTViT
    with torch.no_grad():
        tokens = ct_vit_model(x, return_encoded_tokens=True)

    # Flatten → CPU
    B, T, H, W, D = tokens.shape
    feats = tokens.reshape(B, T*H*W, D).squeeze(0).cpu()
    return feats


In [ ]:
# Load ProjectionHead
import torch.nn as nn

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=4096):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, out_dim),
        )
    def forward(self, x):
        pooled = x.mean(dim=1)
        return self.net(pooled)

proj_head = ProjectionHead(in_dim=512, out_dim=4096).to(device)
proj_state = torch.load(
    "/content/drive/MyDrive/projection_head_500scans_joint.pt",
    map_location=device
)
proj_head.load_state_dict(proj_state)
proj_head.eval()
print(" Loaded ProjectionHead")


Load tokenizer, base, and LoRA adapter

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True, padding_side="left"
)
tokenizer.pad_token_id = tokenizer.eos_token_id

base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base,
    "/content/drive/MyDrive/ct_lora_500scans",
    device_map="auto"
)
model.eval()
print(" Loaded LLaMA-8B + LoRA adapter")


Downloading a CT scan for illustration

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

target_folder = "/content"
os.makedirs(target_folder, exist_ok=True)

ct_cached_path = hf_hub_download(
    repo_id="ibrahimhamamci/CT-RATE",
    filename="dataset/........nii.gz",
    repo_type="dataset",
    cache_dir="."
)

# Keep original file name
ct_file_name = os.path.basename(ct_cached_path)
ct_target_path = os.path.join(target_folder, ct_file_name)
shutil.copy(ct_cached_path, ct_target_path)


print(f".........nii.gz saved to: {ct_target_path}")


In [ ]:
def generate_report(scan_path, contrast_used=False, scan_region="thorax"):
    Extract & project
    feats = extract_ctvit_features(scan_path).unsqueeze(0).to(device)
    proj  = proj_head(feats)  # [1,4096]

    contrast = "non-contrast" if not contrast_used else "contrast-enhanced"
    prompt = f"""

Generate a CT report with “Findings” and “Impressions”:

"""
    toks     = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = toks.input_ids.to(device)
    attn_mask = toks.attention_mask.to(device)

    # Build inputs_embeds
    text_embs   = model.get_input_embeddings()(input_ids)          # [1,L,dim]
    proj_emb    = proj.unsqueeze(1).to(text_embs.dtype)            # [1,1,dim]
    inputs_embs = torch.cat([proj_emb, text_embs], dim=1)          # [1,L+1,dim]
    full_mask   = torch.cat([torch.ones(1,1,device=device), attn_mask], dim=1)

    # Decode with tuned sampling & no-repeat
    out = model.generate(
        inputs_embeds=inputs_embs,
        attention_mask=full_mask,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    # Return the completed report
    return tokenizer.decode(out[0], skip_special_tokens=True)


**Flask API for File Upload and Diagnosis Report Generation**

This setup allows you to run the model via a Flask API, making it possible to handle file uploads and generate reports remotely. It’s especially useful if your local machine lacks the necessary computational power, GPU, or RAM for intensive tasks, as the processing can be offloaded to a more capable server through ngrok. However, for a more scalable and reliable solution, deploying the application on cloud services is generally the best approach.

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import os

app = Flask(__name__)
ngrok.set_auth_token("#")

@app.route("/upload", methods=["POST"])
def upload():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400

    file = request.files['file']
    filename = file.filename
    filepath = f"/content/{filename}"
    file.save(filepath)

    print("Received file:", filename)

    diagnosis = str(generate_report(filepath))

    return jsonify({"result": diagnosis.strip()})

# Start server
public_url = ngrok.connect(5000)
print("App running at:", public_url)
app.run(port=5000)


# Output generated by the LLM:

**Findings**

There are bilateral pleural effusion and minimal atelectasis changes in both lungs. In addition to these findings; On the right, there are ground-glass densities with centriacinar distribution around the bronchi tree, which were evaluated as compatible with Covid pneumonia or similar viral pneumonias. No mass was detected in both lung parenchyma. Trachea and main bronchi are open. No occlusive pathology was detected. Mediastinal structures cannot be evaluated optimally due to lack of IV contrast. As far as can be seen; Heart size and contour have normal appearance. Pericardial effusion was not detected. Thoracic esophagus calibration was normal and no significant tumoral wall thickening was detected. No lymphadenopathy was observed in mediastinum and hilar regions. No pathologically sized lymph nodes were detected. When examined in the bone window; There are metastatic lytic lesions in all bones in the thorax. The vertebral corpus heights are preserved. There are osteolytic metastases in the vertebrae. Degenerative changes are present in the intervertebral discs. Upper abdominal organs included in the sections within the image area appear natural. No space-occupying lesion was detected that could be distinguished from solid. Pleural effusions continue into the costovertebral angles on the left. Left hemidiaphragm thickness increases due to this finding. It may also contribute to the decrease in the cross-sectional area measured at the lower border. Intra-abdominal free fluid loculations were not detected. Bilateral adrenal glands appeared natural and no space-occupying lesion distinguishable by size was detected here. No intraabdominal obstructive process was detected.
There are fractures in the first metatarsal bones of both feet visible outside the examination areas.

**Impression**

Metastatic skeletal disease in multiple thoracic locations, likely related to known prostate cancer. Findings consistent with resolved COVID-19 pneumonia in both upper lobes. Bilateral minimal pleural effusion and atelectasis in the basal lung segments. Mild left-sided pleural collection contributing to increased left hemidiaphragmatic thickness.




# **Acknowledgements:**
The CT visual feature extractor (CT-ViT / CT-CLIP) and the CT-RATE dataset used in this project were created and released by Ibrahim Hamamcı and collaborators. Their weeks-long training on high-performance GPUs and their decision to share the artifacts made this project possible. Original repositories/dataset pages:



CT-CLIP: https://github.com/ibrahimethemhamamci/CT-CLIP


CT-RATE dataset: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE